# Phase 2 Task 1: 5-Fold Cross-Validation by Origin Key

**Objective:** Train models with GroupKFold split by origin_key to get proper confidence intervals

**Date:** 30-09-2026

## ⚠️ IMPORTANT: This is a TRAINING task

- **Minimum scope:** ConvNeXt-Tiny + MobileNetV3 × 5 folds = **10 training runs**
- **Recommended scope:** 5 models × 5 folds = 25 training runs
- Each fold trains a NEW model from scratch

## Background

Current evaluation uses fixed train/val/test split where 14/17 species have only 1 origin in test.
This makes confidence intervals unreliable. Origin-key based CV will show true generalization.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

CWD = Path.cwd()
for i, part in enumerate(CWD.parts):
    if part == 'woodbench':
        ROOT = Path(*CWD.parts[:i+1])
        break
else:
    ROOT = CWD

sys.path.insert(0, str(ROOT))

import torch
import timm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

print(f"ROOT: {ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

In [ ]:
# Training configuration
CONFIG = {
    # Models to train (start with minimum: ConvNeXt + MobileNet)
    'models': ['convnext_tiny', 'mobilenetv3_large'],
    # Alternative: train all 5 models
    # 'models': ['convnext_tiny', 'swin_tiny', 'efficientnet_b3', 'mobilenetv3_large', 'resnet50'],
    
    'n_folds': 5,
    'epochs': 30,  # Shorter for CV (faster iteration)
    'batch_size': 32,
    'backbone_lr': 1e-5,
    'head_lr': 1e-4,
    'weight_decay': 1e-4,
    'label_smoothing': 0.1,
    'seed': 42,
    'num_workers': 4,
}

DATA_DIR = ROOT / "data" / "task-B-group-split"
OUTPUT_DIR = ROOT / "data" / "experiments" / "Task-B-new" / "origin_cv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")
print(f"\nOutput: {OUTPUT_DIR}")

## 3. Analyze Origin Distribution

In [ ]:
from sklearn.model_selection import GroupKFold

def extract_origin_key(filename):
    """Extract origin key from filename."""
    # Format: "3351. Araucaria araucana.1.(12).png"
    # Origin: "3351. Araucaria araucana.1"
    if ' (' in filename:
        return filename.rsplit(' (', 1)[0]
    return filename.rsplit('.png', 1)[0] if '.png' in filename else filename

# Collect all images with origin keys
samples = []
for split in ['train', 'val', 'test']:
    split_dir = DATA_DIR / split
    if split_dir.exists():
        for species_dir in sorted(split_dir.iterdir()):
            if species_dir.is_dir():
                species = species_dir.name
                for img_path in species_dir.glob("*.png"):
                    samples.append({
                        'path': str(img_path.relative_to(ROOT)),
                        'filename': img_path.name,
                        'species': species,
                        'split': split,
                        'origin_key': extract_origin_key(img_path.name),
                        'full_path': str(img_path)
                    })

df = pd.DataFrame(samples)
print(f"Total samples: {len(df)}")
print(f"Unique origins: {df['origin_key'].nunique()}")
print(f"Species: {df['species'].nunique()}")

# Origins per species
origins_per_species = df.groupby('species')['origin_key'].nunique().sort_values()
print("\n=== Origins per species ===")
print(origins_per_species)

single_origin = (origins_per_species == 1).sum()
print(f"\n⚠️  Species with only 1 origin: {single_origin}/17")
print("These species will appear in every fold's validation set.")

## 4. Define Dataset and Transforms

In [ ]:
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

class OriginDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

# Transforms
train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(90),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Dataset classes defined.")

## 5. Model Factory

In [ ]:
def create_model(backbone_name, num_classes):
    """Create model from timm."""
    model = timm.create_model(backbone_name, pretrained=True, num_classes=num_classes)
    return model

def setup_training(model, backbone_lr, head_lr, weight_decay, label_smoothing):
    """Setup optimizer, scheduler, criterion."""
    # Differential learning rates
    optimizer = torch.optim.AdamW([
        {'params': model.backbone.parameters() if hasattr(model, 'backbone') else model.parameters(), 
         'lr': backbone_lr},
        {'params': model.head.parameters() if hasattr(model, 'head') else [], 
         'lr': head_lr},
    ], weight_decay=weight_decay)
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])
    criterion = torch.nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    
    return optimizer, scheduler, criterion

print("Model factory defined.")

## 6. Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in tqdm(loader, desc='Training', leave=False):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Evaluating', leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    return total_loss / total, correct / total

print("Training functions defined.")

## 7. Run 5-Fold CV Training

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Get unique origins for GroupKFold
unique_origins = df['origin_key'].unique()
n_folds = min(CONFIG['n_folds'], len(unique_origins))
print(f"Running {n_folds}-fold CV with GroupKFold by origin_key")

# Create class mapping
classes = sorted(df['species'].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
num_classes = len(classes)

results = []

gkf = GroupKFold(n_splits=n_folds)

for model_name in CONFIG['models']:
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")
    
    model_results = []
    
    for fold, (train_origins_idx, val_origins_idx) in enumerate(gkf.split(unique_origins, groups=unique_origins)):
        print(f"\n--- Fold {fold+1}/{n_folds} ---")
        
        train_origins = set(unique_origins[train_origins_idx])
        val_origins = set(unique_origins[val_origins_idx])
        
        print(f"Train origins: {len(train_origins)}, Val origins: {len(val_origins)}")
        
        # Get samples for this fold
        train_mask = df['origin_key'].isin(train_origins)
        val_mask = df['origin_key'].isin(val_origins)
        
        train_df = df[train_mask]
        val_df = df[val_mask]
        
        print(f"Train samples: {len(train_df)}, Val samples: {len(val_df)}")
        
        # Create datasets
        train_dataset = OriginDataset(
            [str(ROOT / p) for p in train_df['path']],
            [class_to_idx[s] for s in train_df['species']],
            transform=train_transform
        )
        val_dataset = OriginDataset(
            [str(ROOT / p) for p in val_df['path']],
            [class_to_idx[s] for s in val_df['species']],
            transform=val_transform
        )
        
        train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, 
                                num_workers=CONFIG['num_workers'], pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
                               num_workers=CONFIG['num_workers'], pin_memory=True)
        
        # Create model
        if model_name == 'convnext_tiny':
            model = timm.create_model('convnext_tiny', pretrained=True, num_classes=num_classes)
        elif model_name == 'mobilenetv3_large':
            model = timm.create_model('mobilenetv3_large_100', pretrained=True, num_classes=num_classes)
        elif model_name == 'swin_tiny':
            model = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=num_classes)
        elif model_name == 'efficientnet_b3':
            model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=num_classes)
        elif model_name == 'resnet50':
            model = timm.create_model('resnet50', pretrained=True, num_classes=num_classes)
        
        model = model.to(device)
        
        # Setup training
        backbone_params = []
        head_params = []
        for name, param in model.named_parameters():
            if 'head' in name or 'classifier' in name or 'fc' in name:
                head_params.append(param)
            else:
                backbone_params.append(param)
        
        optimizer = torch.optim.AdamW([
            {'params': backbone_params, 'lr': CONFIG['backbone_lr']},
            {'params': head_params, 'lr': CONFIG['head_lr']},
        ], weight_decay=CONFIG['weight_decay'])
        
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])
        criterion = torch.nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])
        
        # Train
        best_val_acc = 0
        for epoch in range(CONFIG['epochs']):
            train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
            val_loss, val_acc = evaluate(model, val_loader, criterion, device)
            scheduler.step()
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                # Save best model for this fold
                fold_dir = OUTPUT_DIR / f"{model_name}_fold{fold+1}"
                fold_dir.mkdir(parents=True, exist_ok=True)
                torch.save(model.state_dict(), fold_dir / "best.pth")
            
            if (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1:2d}: Train={train_acc:.4f}, Val={val_acc:.4f}")
        
        print(f"  Best Val Acc: {best_val_acc:.4f}")
        
        model_results.append({
            'model': model_name,
            'fold': fold + 1,
            'best_val_acc': best_val_acc,
            'train_samples': len(train_df),
            'val_samples': len(val_df),
        })
        
        # Clean up GPU memory
        del model
        torch.cuda.empty_cache()
    
    results.extend(model_results)

print(f"\n{'='*60}")
print("Training complete!")
print(f"{'='*60}")

## 8. Summarize Results

In [ ]:
results_df = pd.DataFrame(results)

print("\n=== 5-FOLD CV RESULTS (by Origin Key) ===")
print(results_df.to_string(index=False))

print("\n=== Summary Statistics ===")
summary = results_df.groupby('model').agg({
    'best_val_acc': ['mean', 'std', 'min', 'max']
}).round(4)
print(summary)

# Compare with original fixed split
print("\n=== Comparison: Origin CV vs Fixed Split ===")
fixed_split = {
    'convnext_tiny': 0.9956,
    'mobilenetv3_large': 0.9837,
}

for model in results_df['model'].unique():
    cv_mean = results_df[results_df['model']==model]['best_val_acc'].mean()
    cv_std = results_df[results_df['model']==model]['best_val_acc'].std()
    fixed_acc = fixed_split.get(model, 0)
    
    print(f"\n{model}:")
    print(f"  Origin CV: {cv_mean:.4f} ± {cv_std:.4f}")
    print(f"  Fixed split: {fixed_acc:.4f}")
    print(f"  Gap: {(fixed_acc - cv_mean)*100:+.2f}%")

## 9. Conclusions

In [ ]:
print("""
============================================================
5-FOLD CV BY ORIGIN KEY - CONCLUSIONS
============================================================

Key Findings:
1. True generalization: CV accuracy shows how model performs on unseen origins
2. Confidence intervals: ± std gives reliable uncertainty estimate
3. Gap analysis: If CV << fixed split, original results were optimistic

Implications:
- If gap is small: Origin-key split is similar to random split
- If gap is large: Model was benefiting from seeing same origins
- Use CV results for paper's confidence intervals

============================================================
""")

# Save results
results_df.to_csv(OUTPUT_DIR / 'cv_results.csv', index=False)
summary.to_csv(OUTPUT_DIR / 'cv_summary.csv')

import json
with open(OUTPUT_DIR / 'cv_config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f"Results saved to: {OUTPUT_DIR}")